# Viscosity correction (HI 9.6.7 / ISO TR 17766)

**Use case:** performance on a viscous liquid predicted from a water test.  This is
the worked example of the *Correction* step of the workbench.

At the water best-efficiency point (BEP-W), with $\nu$ in cSt, $H$ in ft per stage,
$Q$ in gpm and $N$ in rpm:

$$ B = \frac{26.6\,\nu^{0.5}\,H^{0.0625}}{Q^{0.375}\,N^{0.25}},\quad
C_Q = 2.71^{-0.165\,\log_{10}(B)^{3.15}},\quad
C_H = 1-(1-C_Q)\Big(\frac{Q}{Q_{BEP}}\Big)^{0.75},\quad
C_\eta = B^{-0.0547\,B^{0.69}} $$

Valid for $1 < B < 40$; for $B \le 1$ there is no correction.  `pump.viscosity` provides
`b_parameter` and `correction_factors`; `PerformanceCurve.bep()` and
`PerformanceCurve.to_viscous` apply them to a whole curve (every point becomes
$Q\,C_Q$, $H\,C_H(Q/Q_{BEP})$, $\eta\,C_\eta$ and the power is recomputed with the
viscous density).

> The equations are ANSI/HI 9.6.7-2010 Eq. 2-7 as reproduced by a **secondary source**.
> They were cross-checked independently but **not** against the standard text.

**Acceptance:** B and the factors match a hand computation; the viscous BEP moves by
$C_Q$ and $C_\eta$; B <= 1 is bit-identical to a density-only change; B >= 40 warns.

## 1 · Setup and the water test

In [ ]:
import warnings

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

from pump import Q_, Fluid, Water, TestPoint, PerformanceCurve

# Illustrative water test of pump P-1001A (1480 rpm):
#          Q [m3/h]  Head [m]  P [kW]  eta [%]
P1001A = [
    (160.0, 125.37, 77.97, 70.03),
    (240.0, 119.11, 95.17, 81.82),
    (320.0, 110.79, 114.44, 84.44),
    (400.0, 100.01, 147.26, 74.00),
    (480.0, 86.72, 173.98, 65.13),
    (540.0, 75.60, 194.76, 57.14),
]
water = Fluid("Test water", density=Q_(1000.0, "kg/m**3"))


def build_curve(rows, fluid, speed, degree=3, head_k=1.0, power_k=1.0, eff_k=1.0):
    "Measured rows -> PerformanceCurve with Head / eta pinned on the points."
    points = [
        TestPoint(
            fluid=fluid,
            capacity=Q_(q, "m**3/h"),
            speed_of_rotation=speed,
            _head=Q_(h * head_k, "m"),
            _efficiency=Q_(e * eff_k, "percent"),
            breaking_power=Q_(p * power_k, "kW"),
        )
        for q, h, p, e in rows
    ]
    return PerformanceCurve(fluid, points, polynomial_degree=degree)

from pump import b_parameter, correction_factors

speed = Q_(1480, "rpm")
water_curve = build_curve(P1001A, water, speed)
q_bep, h_bep, eta_bep = water_curve.bep()
print(f"BEP (water): Q = {q_bep:.1f~P}  H = {h_bep:.1f~P}  eta = {eta_bep:.1f~P}")

## 2 · B and the factors for a 100 cSt oil

The liquid carries its density and kinematic viscosity (any unit; here 100 cSt).
`b_parameter` takes Pint quantities and converts to the US units of the equation.

In [ ]:
oil = Fluid("Lube oil", density=Q_(880.0, "kg/m**3"), kinematic_viscosity=Q_(100.0, "cSt"))
nu = oil.kinematic_viscosity

b = b_parameter(nu, q_bep, h_bep, speed)
c_q, c_h, c_eta = correction_factors(b)   # at the BEP: C_H = C_Q
print(f"B = {b:.3f}  C_Q = {c_q:.4f}  C_H(BEP) = {c_h:.4f}  C_eta = {c_eta:.4f}")

# the same B by hand, in the US units of the equation
nu_cst = nu.to("cSt").magnitude
q_gpm = q_bep.to("gallon/minute").magnitude
h_ft = h_bep.to("ft").magnitude
n_rpm = speed.to("rpm").magnitude
b_hand = 26.6 * nu_cst**0.5 * h_ft**0.0625 / (q_gpm**0.375 * n_rpm**0.25)
print(f"B by hand = {b_hand:.3f}")

## 3 · The corrected curve

`to_viscous` returns a **new** curve and stores the BEP factors in `viscosity_correction`.

In [ ]:
viscous = water_curve.to_viscous(oil)
vc = viscous.viscosity_correction
print({k: round(v, 4) for k, v in vc.items()})

q_v, h_v, eta_v = viscous.bep()
print(f"BEP (viscous): Q = {q_v:.1f~P}  H = {h_v:.1f~P}  eta = {eta_v:.1f~P}")

fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 8))
for curve_, label in ((water_curve, "water"), (viscous, "oil 100 cSt")):
    caps = np.asarray(curve_.fitter.capacities, dtype=float)
    xs = np.linspace(caps.min(), caps.max(), 150)
    axes[0].plot(xs, np.polyval(curve_.fitter.head_coeffs, xs), label=label)
    axes[1].plot(xs, np.polyval(curve_.fitter.power_coeffs, xs))
    axes[2].plot(xs, np.polyval(curve_.fitter.efficiency_coeffs, xs))
axes[0].set_ylabel("Head (m)")
axes[1].set_ylabel("Power (kW)")
axes[2].set_ylabel("Efficiency (%)")
axes[2].set_xlabel("Capacity (m3/h)")
axes[0].legend()
for ax in axes:
    ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 4 · Edge cases

Hot water (B <= 1) is *not* corrected: `to_viscous` equals a density-only `to_fluid`.
Two stages halve the head per stage in B.  B >= 40 is outside the method and warns.

In [ ]:
hot = Water(Q_(60, "degC"))
low_b = water_curve.to_viscous(hot)
density_only = water_curve.to_fluid(hot)

two_stage = water_curve.to_viscous(oil, stages=2)

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    correction_factors(45.0)
warned = any(issubclass(w.category, UserWarning) for w in caught)

print("B(hot water, 60 degC) =", round(low_b.viscosity_correction["B"], 3))
print("B with 2 stages       =", round(two_stage.viscosity_correction["B"], 3))
print("B = 45 warns          =", warned)

## 5 · Acceptance

In [ ]:
# B and the factors
assert 1.0 < b < 40.0
assert abs(b - b_hand) < 1e-12
assert abs(vc["B"] - b) < 1e-12
assert (vc["C_Q"], vc["C_H"], vc["C_eta"]) == (c_q, c_h, c_eta)
assert 0.0 < c_eta < c_q <= 1.0

# the viscous BEP moves by C_Q (flow, head) and C_eta (efficiency)
assert abs(q_v.magnitude / q_bep.magnitude - c_q) < 0.01
assert abs(h_v.magnitude / h_bep.magnitude - c_q) < 0.02
assert abs(eta_v.magnitude / eta_bep.magnitude - c_eta) < 0.02

# shut-off head is not corrected: C_H(Q=0) = 1, continuity at B = 1
assert correction_factors(5.0, 0.0)[1] == 1.0
assert max(abs(x - 1.0) for x in correction_factors(1.0 + 1e-9)) < 1e-6
assert correction_factors(0.5) == (1.0, 1.0, 1.0)

# B <= 1 is bit-identical to a density-only change
assert low_b.viscosity_correction["B"] <= 1.0
q_test = Q_(320, "m**3/h")
assert low_b.predict_head(q_test).magnitude == density_only.predict_head(q_test).magnitude
assert low_b.predict_breaking_power(q_test).magnitude == density_only.predict_breaking_power(q_test).magnitude

# stages lower B; the original curve is untouched; B >= 40 warns
assert two_stage.viscosity_correction["B"] < vc["B"]
assert water_curve.viscosity_correction is None
assert warned
print("Viscosity HI 9.6.7 acceptance OK")